In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
import random
from datetime import datetime, timedelta
from stonesoup.models.transition.linear import CombinedLinearGaussianTransitionModel, ConstantVelocity
from stonesoup.types.groundtruth import GroundTruthState
from stonesoup.simulator.simple import SingleTargetGroundTruthSimulator

np.random.seed(1990)
random.seed(1990)

start_time = datetime.now().replace(microsecond=0)

# Generate transition model
# i.e. fk(xk|xk-1)
transition_model = CombinedLinearGaussianTransitionModel([ConstantVelocity(0.005),
                                                          ConstantVelocity(0.005)])

time_max = 180  # timestamps the simulation is observed over
timesteps = [start_time + timedelta(seconds=k) for k in range(time_max)]

xdirection = 1
ydirection = 1

ground_truth_simulator = SingleTargetGroundTruthSimulator(
    transition_model=transition_model,
    initial_state=GroundTruthState([-50, 1, -50, 1], timestamp=timesteps[0]),
    number_steps=time_max,
    timestep=timedelta(seconds=1),
)

ground_truth = list(ground_truth_simulator)




# Load Stone Soup materials
from stonesoup.types.state import State
from stonesoup.types.array import StateVector, CovarianceMatrix

from stonesoup.platform.base import FixedPlatform 
import stonesoup.sensor.radar.radar as radar
RadarRotatingBearing = radar.RadarRotatingBearing
RadarRotatingBearingRange = radar.RadarRotatingBearingRange
RadarRasterScanBearingRange = radar.RadarRasterScanBearingRange
from stone_soup_extension.automatically_rotating_radar import RadarBearingRange_automatically_rotating
from stonesoup.movable.grid import NStepDirectionalGridMovable


from stonesoup.types.state import State

# setup the sensors

# setup the transmitter in same place as the norfolk park field experiment

transmitter_xx = [ -44.56495493,   17.54448279, -132.44165643,  127.58969322]
transmitter_yy = [-106.13383274,  207.6112732,   207.36738564,  -76.98800717]

from typing import List
sensor_platforms: List[FixedPlatform] = []

platform = FixedPlatform(
    State([transmitter_xx[0], transmitter_yy[0]]),  
    position_mapping=[0, 1],
)

# Configure the radar noise, since we are using just a single dimension we need to specify only the
# noise associated with the bearing dimension, we assume a bearing accuracy of +/- 0.025 degrees for
# each measurement
noise_covar = CovarianceMatrix(np.array(np.diag([np.deg2rad(0.025) ** 2])))

# instantiate the radar
# sensor = RadarRasterScanBearingRange(
#     position_mapping=(0, 1),
#     noise_covar=noise_covar,
#     #position=[[0], [0]],
#     rpm=1,
#     for_angle=np.radians(360),
#     fov_angle=np.radians(30),
#     dwell_centre=StateVector([np.pi]),
#     max_range=500
    
#     #clutter_model=None,
#     #resolution=np.radians(res)  # resolution of sensor equal to distance between search cells
# )

sensor = RadarRotatingBearingRange(
    position_mapping=(0, 2),
    noise_covar=np.array([[np.radians(1)**2, 0],
                          [0, 1**2]]),
    ndim_state=4,
    rpm=1,
    fov_angle=np.radians(30),
    dwell_centre=StateVector([np.radians(90)]),
    max_range=800,
    resolution=Angle(np.radians(30)))

sensor.timestamp = start_time
platform.add_sensor(sensor)

sensor_platforms.append(platform)

from stonesoup.types.angle import Angle



# platform = FixedPlatform(
#     movement_controller=NStepDirectionalGridMovable(states=[State([[0], [0]],
#                                                                   timestamp=start_time)],
#                                                     position_mapping=(0, 1),
#                                                     resolution=1,
#                                                     n_steps=2,
#                                                     step_size=6.25,  # 6.25 seems to match target
#                                                     action_mapping=(0, 1)),
#     sensors=[sensor])


ModuleNotFoundError: No module named 'stone_soup_extension'

In [1]:
import copy
from collections import defaultdict
sensor_history = defaultdict(dict)
import importlib
importlib.reload(radar)

plotter.fig.data = []

# plotting sensor?
for ts in timesteps:
    sensor.act(ts)
    sensor_history[ts][sensor] = copy.deepcopy(sensor)

# Using Plotterly
from stonesoup.plotter import AnimatedPlotterly

plotter = AnimatedPlotterly(timesteps, tail_length=0.1)

plotter.plot_ground_truths(ground_truth[0][1], [0,2])
plotter.plot_sensors(platform)

plotter.plot_sensor_fov({sensor}, sensor_history)
plotter.fig.show()


NameError: name 'radar' is not defined